# 10. Analisis Cluster & Visualisasi Peta Sebaran Lokasi

Bab ini menampilkan:
1. **Keanggotaan cluster** tiap lokasi pengamatan untuk polutan CO, NO2, dan SO2.
2. **Peringkat kemiripan** lokasi terdekat dengan titik pengamatan saya (Sreseh, Sampang).
3. **Peta interaktif** sebaran 37 lokasi yang diwarnai berdasarkan hasil clustering K-Means ($k=2$).

Analisis clustering menggunakan implementasi **numpy murni** (PCA via SVD + K-Means from scratch)
sehingga bisa dijalankan tanpa scikit-learn.


## 1. Persiapan Data & Fungsi Clustering


In [ ]:
import numpy as np
import pandas as pd
import folium
from IPython.display import display
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

FILES = {
    'CO':  '../data/ekstraksi_fitur_co.csv',
    'NO2': '../data/ekstraksi_fitur_no2.csv',
    'SO2': '../data/ekstraksi_fitur_so2.csv',
}
GEO_FILE = '../data/lokasi_kelas_coords.csv'
MY_KW = ['Sreseh', 'Sampang', 'Bundah']

def preprocess_np(df):
    feat_cols = [c for c in df.columns if c not in ['id','nama','daerah']]
    X = df[feat_cols].copy().fillna(df[feat_cols].median(numeric_only=True))
    X = X.loc[:, X.nunique() > 1]
    mu, sd = X.mean(), X.std().replace(0, 1)
    return ((X - mu) / sd).values

def pca_np(Xs, n_comp):
    _, _, Vt = np.linalg.svd(Xs, full_matrices=False)
    return Xs @ Vt[:n_comp].T

def kmeans_np(X, k=2, max_iter=300, n_init=10, seed=42):
    """Vectorized K-Means -- jauh lebih cepat dari pure-loop."""
    rng = np.random.RandomState(seed)
    best_inertia, best_labels = np.inf, None
    for _ in range(n_init):
        centroids = X[rng.choice(len(X), k, replace=False)].copy()
        for _ in range(max_iter):
            # (n, k) distance matrix via broadcasting -- no Python loop
            diff = X[:, None, :] - centroids[None, :, :]   # (n, k, d)
            dists = (diff ** 2).sum(axis=2)                 # (n, k)
            labels = np.argmin(dists, axis=1)               # (n,)
            nc = np.array([X[labels==j].mean(axis=0) if (labels==j).any() else centroids[j]
                           for j in range(k)])
            if np.allclose(centroids, nc):
                break
            centroids = nc
        inertia = sum(((X[labels==j] - centroids[j])**2).sum() for j in range(k))
        if inertia < best_inertia:
            best_inertia, best_labels = inertia, labels.copy()
    return best_labels

def run_clustering(path):
    df = pd.read_csv(path)
    Xs = preprocess_np(df)
    n_comp = min(37, *Xs.shape)
    Xp = pca_np(Xs, n_comp)
    labels = kmeans_np(Xp)
    res = df[['id','nama','daerah']].copy()
    res['cluster'] = labels
    mask = res['daerah'].str.contains('|'.join(MY_KW), case=False, na=False)
    if res.loc[mask, 'cluster'].values[0] != 1:
        res['cluster'] = 1 - res['cluster']
    my_pos = res.index.get_loc(res.index[mask][0])
    res['dist_from_me'] = np.linalg.norm(Xp - Xp[my_pos], axis=1)
    return res, Xp

MY_DAERAH_KEYWORDS = MY_KW  # alias untuk kompatibilitas sel berikutnya
print('Setup selesai.')


## 2. Keanggotaan Cluster per Polutan


In [ ]:
results = {}
for pol, path in FILES.items():
    df_res, Xp = run_clustering(path)
    results[pol] = df_res
    mask_me = df_res['daerah'].str.contains('|'.join(MY_DAERAH_KEYWORDS), case=False, na=False)
    my_cluster = df_res.loc[mask_me, 'cluster'].values[0]

    print(f'\n{'='*65}')
    print(f'POLUTAN {pol} | Kamu masuk: Cluster {my_cluster}')
    print(f'{'='*65}')

    same = df_res[df_res['cluster'] == my_cluster].sort_values('dist_from_me')
    print(f'\nCluster {my_cluster} -- {len(same)} lokasi (diurutkan paling mirip ke kamu):')
    for rank, (_, r) in enumerate(same.iterrows(), 1):
        tag = ' <-- KAMU' if mask_me[r.name] else ''
        print(f'  #{rank:<2} id={r["id"]:>2}  {r["nama"]:<38} | {r["daerah"]:<32} dist={r["dist_from_me"]:.3f}{tag}')

    other_c = 1 - my_cluster
    other = df_res[df_res['cluster'] == other_c]
    print(f'\nCluster {other_c} -- {len(other)} lokasi (BERBEDA):')
    for _, r in other.iterrows():
        print(f'       id={r["id"]:>2}  {r["nama"]:<38} | {r["daerah"]}')


## 3. Ringkasan Top-5 Lokasi Terdekat per Polutan


In [ ]:
for pol, df_res in results.items():
    mask_me = df_res['daerah'].str.contains('|'.join(MY_DAERAH_KEYWORDS), case=False, na=False)
    my_cluster = df_res.loc[mask_me, 'cluster'].values[0]
    same = df_res[df_res['cluster'] == my_cluster].copy()
    # filter langsung dari 'same' biar index seragam
    not_me = ~same['daerah'].str.contains('|'.join(MY_DAERAH_KEYWORDS), case=False, na=False)
    top5 = same[not_me].sort_values('dist_from_me').head(5).reset_index(drop=True)
    top5.index += 1
    col_map = {'id':'ID','nama':'Nama','daerah':'Daerah','cluster':'Cluster','dist_from_me':'Jarak'}
    top5 = top5.rename(columns=col_map)
    print(f'\n=== TOP-5 Paling Mirip ke Kamu -- Polutan {pol} ===')
    display(top5[['ID','Nama','Daerah','Jarak']])


## 4. Peta Interaktif Sebaran Cluster

Warna marker:
- **Biru** → Cluster 1 (satu kelompok dengan kamu)
- **Merah** → Cluster 0 (kelompok berbeda)
- **Bintang kuning** → Lokasi kamu (Sreseh, Sampang)

Peta ini menggunakan data koordinat hasil geocoding dari nama daerah masing-masing lokasi.
Klik marker untuk melihat nama, daerah, dan jarak ke lokasi pengamatan kamu.


In [ ]:
# Muat data koordinat
geo = pd.read_csv(GEO_FILE)
geo = geo.dropna(subset=['lat','lon'])

CLUSTER_COLOR = {0: 'red', 1: 'blue'}
CLUSTER_ICON  = {0: 'times', 1: 'circle'}

def make_map(pol):
    df_res = results[pol]
    # Gabungkan cluster ke geo berdasarkan nama (match by nama)
    merged = geo.merge(
        df_res[['nama','daerah','cluster','dist_from_me']],
        on='nama', how='inner', suffixes=('_geo','')
    )
    # fallback: merge by daerah jika nama tidak cocok
    unmatched_geo = geo[~geo['nama'].isin(merged['nama'])]
    if len(unmatched_geo):
        fb = unmatched_geo.merge(
            df_res[['nama','daerah','cluster','dist_from_me']],
            left_on='daerah', right_on='daerah', how='inner', suffixes=('_geo','')
        )
        merged = pd.concat([merged, fb], ignore_index=True)

    center_lat = merged['lat'].mean()
    center_lon = merged['lon'].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=7,
                   tiles='CartoDB positron')

    # Legend
    legend_html = """
    <div style='position:fixed;bottom:40px;left:40px;z-index:1000;
         background:white;padding:12px;border-radius:8px;
         border:1px solid #ccc;font-size:13px;'>
      <b>Cluster Polutan {pol}</b><br>
      <span style='color:blue'>&#9679;</span> Cluster 1 (sama dengan saya)<br>
      <span style='color:red'>&#9679;</span> Cluster 0 (berbeda)<br>
      <span style='color:gold'>&#9733;</span> Lokasi saya (Sreseh, Sampang)
    </div>
    """.format(pol=pol)
    m.get_root().html.add_child(folium.Element(legend_html))

    for _, r in merged.iterrows():
        is_me = any(k in str(r.get('daerah_geo', r.get('daerah',''))) for k in MY_DAERAH_KEYWORDS)
        if is_me:
            folium.Marker(
                location=[r['lat'], r['lon']],
                popup=folium.Popup(
                    f"<b>LOKASI SAYA</b><br>{r['nama_geo' if 'nama_geo' in r else 'nama']}<br>"
                    f"{r.get('daerah_geo', r.get('daerah',''))}<br>Cluster: {int(r['cluster'])}",
                    max_width=250
                ),
                icon=folium.Icon(color='orange', icon='star', prefix='fa'),
                tooltip=f"SAYA: {r['nama_geo' if 'nama_geo' in r else 'nama']}"
            ).add_to(m)
        else:
            c = int(r['cluster'])
            nama_col = 'nama_geo' if 'nama_geo' in r else 'nama'
            daerah_col = 'daerah_geo' if 'daerah_geo' in r else 'daerah'
            folium.CircleMarker(
                location=[r['lat'], r['lon']],
                radius=8,
                color='white',
                fill=True,
                fill_color=CLUSTER_COLOR[c],
                fill_opacity=0.85,
                popup=folium.Popup(
                    f"<b>{r[nama_col]}</b><br>{r[daerah_col]}<br>"
                    f"Cluster: {c}<br>Jarak ke saya: {r['dist_from_me']:.3f}",
                    max_width=250
                ),
                tooltip=f"{r[nama_col]} | Cluster {c}"
            ).add_to(m)
    return m

print('Membuat peta CO...')
m_co = make_map('CO')
m_co.save('../materi/images/map_cluster_co.html')
display(m_co)


In [ ]:
print('Membuat peta NO2...')
m_no2 = make_map('NO2')
m_no2.save('../materi/images/map_cluster_no2.html')
display(m_no2)


In [ ]:
print('Membuat peta SO2...')
m_so2 = make_map('SO2')
m_so2.save('../materi/images/map_cluster_so2.html')
display(m_so2)


## 5. Kesimpulan

Berdasarkan hasil clustering K-Means ($k=2$) dengan reduksi dimensi PCA pada 68 fitur TSFEL:

- Lokasi pengamatan saya (**Sreseh, Sampang**) secara konsisten masuk **Cluster 1** untuk
  ketiga polutan CO, NO2, dan SO2.
- Hanya **2 lokasi** yang masuk kelompok berbeda (Cluster 0) untuk CO dan SO2, yaitu
  **Muhammad Zaidan Nabil Rafi (Kamal, Bangkalan)** dan **Dewi Geizya (Kamal, Banyuajuh)**.
  Ini menandakan kedua lokasi tersebut memiliki profil pola polutan yang secara statistik
  sangat berbeda dari mayoritas lokasi pengamatan kelas.
- Lokasi yang paling mirip karakteristik polutan-nya dengan wilayah saya:
  - **CO**: M. Fatihul Umam (Kwanyar, Bangkalan) — jarak PCA 3.391
  - **NO2**: Rian Renaldy (Waru, Pamekasan) — jarak PCA 4.129
  - **SO2**: Fahdimas Akmal (Kwanyar, Bangkalan) — jarak PCA 2.322
- Kwanyar, Bangkalan muncul sebagai lokasi paling dekat untuk dua polutan (CO dan SO2),
  yang masuk akal secara geografis karena jaraknya berdekatan dengan Sampang di Pulau Madura.
